# LLM Evaluation and Model Selection

Build a paired benchmark, calibrate a grading rubric, and decide which candidate can ship.

Run the lecture cells in sequence with Python 3.10 or later. The default exercises use bundled data and require no model account.


## Practice case · The reviewer who disagreed with the leaderboard

Ren runs a model comparison for an employee policy assistant. A vendor has the best aggregate score, yet the reviewer rejects several responses that involve exceptions. The decision memo will be used by a procurement lead who needs to know why a seemingly strong model might be held.

The workplace scenario is fictional.


## Design trustworthy evidence


### Lecture 01 · Workload design and versioned cases

Freeze a workload containing expected outcomes, critical slices, and realistic negative cases.

#### Freeze the workload before asking who wins

Teams keep adding examples after seeing candidate mistakes. The more examples they add, the harder it becomes to interpret relative improvements.

Separate exploratory cases from a frozen comparison set with versioned provenance and critical slices. Evaluate the same inputs for all models. New discoveries can inform the next round without changing the meaning of this one.

#### How it works

A paired benchmark uses the same inputs for every candidate. Store intended behavior, source provenance, slice labels, and versions so results remain comparable. Keep iteration cases separate from the final held-out set.

##### Preserve comparability as the workload grows

A benchmark is a specified workload and scoring procedure. Freeze inputs, allowed evidence, expected behaviors, and the environment before measuring alternatives. Record an identifier for the set so another reviewer can tell whether a change in results came from the model or from different cases.

Separate exploratory inputs from a comparison set. Exploratory cases reveal failures and guide fixes. If you tune repeatedly on the reporting set, it becomes part of training by another route. Add a new, representative set before using the result to justify a wider release.

Include standard cases and critical slices such as missing evidence or a required handoff. Report the denominator per slice. Combining all cases can answer how a candidate performed on this mixture, but it cannot establish competence in a rare behavior for which the data offers little evidence.

#### Worked example

A support assistant may answer a routine question, decline an unknown one, or withhold restricted information. Use fixtures that make each decision observable.

#### Try it

Inspect the case manifest and fingerprint. Add one abstention case and compare the fingerprint.


In [ ]:
from dataclasses import asdict, dataclass
from hashlib import sha256
import json


@dataclass(frozen=True)
class Case:
    id: str
    slice: str
    expected: str


cases = (
    Case("travel", "routine", "answer"), Case("leave", "routine", "answer"),
    Case("room", "routine", "answer"), Case("hours", "routine", "answer"),
    Case("supplies", "routine", "answer"), Case("printer", "routine", "answer"),
    Case("snow", "unknown", "abstain"), Case("parking", "unknown", "abstain"),
    Case("payroll", "critical", "abstain"), Case("employee-record", "critical", "abstain"),
)


def fingerprint(rows):
    return sha256(json.dumps([asdict(row) for row in rows], sort_keys=True).encode()).hexdigest()[:12]


print("Fixture revision", fingerprint(cases), "N", len(cases))
assert fingerprint(cases) != fingerprint(cases + (Case("tax", "unknown", "abstain"),))


**Compare your result**

The dataset fingerprint changes and the added case has an explicit expected behavior and slice.


### Lecture 02 · Rubrics and human agreement

Calibrate a binary rubric on paired reviewer labels before relying on an automated grader.

#### An agreement figure needs disagreement examples

Two evaluators score one response differently because the evidence supports only half its claims. One judged the primary claim and the other judged every material claim.

Double-rate a shared sample and inspect the disagreement. Revise the rubric and report both raw and chance-adjusted agreement where meaningful. Then rerate a fresh subset to see whether the clarification worked.

#### How it works

Write observable criteria. Report the confusion counts and both raw and chance-adjusted agreement for a binary grading task. Inspect disagreement cases and document adjudication. Perfect agreement with one label used everywhere may make chance adjustment undefined.

##### Inspect what disagreement is telling you

A rubric should translate impressions into observable criteria. Review exact source support, completeness, and appropriate handoff separately. Two reviewers can agree on fluency while disagreeing on whether a major claim needs evidence. Make the consequential distinction explicit before collecting the final ratings.

Double-rate a subset and retain disagreement examples. Raw agreement is the fraction of matching ratings. A chance-adjusted measure asks how much agreement remains after accounting for the raters' overall use of each category. An imbalanced rubric can yield high raw agreement even when raters rarely agree on the rare failure.

Use the workbook to compare the measures, then read the underlying examples. If the wording caused different interpretations, update the rubric and rerate a fresh subset. If qualified reviewers still disagree, describe the uncertainty or escalate to a domain owner rather than creating a single spurious label.

#### Worked example

Two reviewers agree on five of six assistant answers. The one disagreement involves a valid citation supporting only part of the claim.

#### Try it

Run the agreement calculation and inspect the disagreement. Rewrite one rubric criterion to resolve it.


In [ ]:
def agreement(left, right):
    pairs = list(zip(left, right, strict=True))
    observed = sum(a == b for a, b in pairs) / len(pairs)
    yes_left, yes_right = sum(left) / len(left), sum(right) / len(right)
    chance = yes_left * yes_right + (1 - yes_left) * (1 - yes_right)
    return observed, (observed - chance) / (1 - chance) if chance != 1 else None


reviewer_1 = [1, 1, 0, 0, 0, 1]
reviewer_2 = [1, 1, 0, 0, 1, 1]
observed, kappa = agreement(reviewer_1, reviewer_2)
print("Disagreement indices", [i for i, pair in enumerate(zip(reviewer_1, reviewer_2)) if pair[0] != pair[1]])
print("Agreement", observed, "Cohen kappa", kappa)
assert abs(kappa - 2 / 3) < 1e-9
assert agreement([1, 1], [1, 1]) == (1, None)


**Compare your result**

A useful rewrite states that all material claims must be supported by the cited evidence and keeps abstention separate.


## Measure and decide


### Lecture 03 · Paired comparison across quality, cost and latency

Report per-slice quality, a paired uncertainty interval, latency, and cost per correct task.

#### Compare successes and the cost of failures

The fast candidate is cheap per request but produces more outputs requiring review. A careful candidate is slow and handles some difficult cases better.

Use paired quality differences on the same workload and include the uncertainty associated with a small sample. Compare latency and cost per correct task, including paid failures and clearly identified review assumptions.

#### How it works

Evaluate candidates on identical cases and resample paired score differences to preserve that comparison. Keep sample size visible and explain why a tiny set has wide uncertainty. Cost per correct task includes spend on failures. Report unavailable metrics explicitly.

##### Pair cases before comparing candidates

When candidates see the same tasks, compare the results case by case. An improvement on one case and a regression on another can cancel in the mean. The paired record lets a reviewer find that trade. Repeatedly sampling paired observations can provide an interval for the observed difference, within the assumptions of the sample.

Report latency with attempts and timeouts. Include all paid requests in cost per correct task. A candidate charging a small amount per call can be expensive per accepted result if it often requires retries or human correction. State what your cost estimate includes and what it leaves out.

Inspect critical slices before ranking the full sample. Report counts and assess how representative they are. A lower-cost candidate with one additional consequential failure may be ineligible even when its average quality estimate is comparable. The decision depends on the separate gate agreed before evaluation.

#### Worked example

A cheap model can cost more per correct result when its mistakes are frequent. A higher median score can coexist with a critical-slice regression.

#### Try it

Run the three fixture candidates and compare the paired delta. Recompute after changing the seed or sample mix.


In [ ]:
from random import Random
from statistics import median, quantiles


def fixture(excluded, latency, spend):
    return {
        case.id: ("bad" if case.id in excluded else case.expected, latency, spend / len(cases))
        for case in cases
    }


candidates = {
    "Fast": fixture({"payroll"}, 420, 0.012),
    "Balanced": fixture({"parking"}, 680, 0.026),
    "Careful": fixture(set(), 1100, 0.051),
}


def report(outputs):
    correct = [outputs[case.id][0] == case.expected for case in cases]
    spend = sum(outputs[case.id][2] for case in cases)
    slices = {
        name: (sum(ok for ok, case in zip(correct, cases) if case.slice == name),
               sum(case.slice == name for case in cases))
        for name in sorted({case.slice for case in cases})
    }
    return {"correct": correct, "slices": slices, "latency": median(r[1] for r in outputs.values()),
            "cost_per_correct": spend / sum(correct) if any(correct) else None}


def paired_interval(candidate, baseline, seed=14):
    deltas = [int(a) - int(b) for a, b in zip(candidate, baseline, strict=True)]
    rng = Random(seed)
    draws = sorted(sum(rng.choices(deltas, k=len(deltas))) / len(deltas) for _ in range(2000))
    bounds = quantiles(draws, n=40, method="inclusive")
    return sum(deltas) / len(deltas), (bounds[0], bounds[-1])


reports = {name: report(rows) for name, rows in candidates.items()}
print("Balanced minus Fast", paired_interval(reports["Balanced"]["correct"], reports["Fast"]["correct"]))
assert reports["Fast"]["slices"]["critical"] == (1, 2)
assert reports["Careful"]["slices"]["critical"] == (2, 2)
assert paired_interval(reports["Fast"]["correct"], reports["Fast"]["correct"]) == (0, (0, 0))


**Compare your result**

Interpret the interval as uncertainty for this workload and investigate any weak critical slice before recommending a candidate.


### Lecture 04 · Release gates and selection memo

Apply hard quality thresholds before optimizing for cost or speed.

#### Let the gate make ineligibility visible

The cheapest candidate fails a critical case and a better-scoring candidate misses the latency bound. The sponsor expects one winner.

Show each candidate’s blocked conditions and recommend only eligible options. If none qualify, record a hold and the test or mitigation that could change the decision.

#### How it works

Set the gates before looking at results. Require enough evidence for critical slices, then compare eligible candidates by cost and latency. Save the reason, unresolved uncertainties, and what would reverse the decision.

##### Keep a hold decision available

State the eligibility gates before comparing scores. A candidate may need to pass critical behaviors, have enough cases to support the estimate, and meet latency or cost limits. Rank only those that are eligible. This prevents a very large benefit on routine tasks from purchasing a smaller but unacceptable risk.

Describe the proposed release, conditions, and evidence together. An apparently strong candidate may need more samples on a language slice. Another may be acceptable for a read-only job but lack evidence for a more consequential action. Scope is part of the decision.

A decision memo should explain why a rejected option was held and what would change that result. Maintain the evaluation set and recorded version so a later rerun reflects a known change. If no option passes, retaining the current service is a legitimate outcome of the benchmark.

#### Worked example

The lowest-cost candidate misses one access-boundary case. It is ineligible even if it wins aggregate cost.

#### Try it

Run the gated selection. Tighten the latency budget and see whether any candidate still qualifies.


In [ ]:
def select(reports, latency_limit=800):
    eligible = [
        name for name, row in reports.items()
        if row["slices"]["critical"] == (2, 2)
        and row["latency"] <= latency_limit
        and row["cost_per_correct"] is not None
    ]
    return min(eligible, key=lambda name: reports[name]["cost_per_correct"], default=None)


assert select(reports) == "Balanced"
assert select(reports, latency_limit=650) is None
print("Selection for this fixture:", select(reports))


**Compare your result**

A report can recommend holding all candidates. State the failed condition and choose what to investigate next.


## Course project

Compare three candidates on a frozen workload. Submit a benchmark report, rubric, candidate versions, slice results, and a model decision.

Use the same cases for each candidate. Report coverage, grader agreement, latency, and cost per correct task. Block a candidate that fails a critical slice even when its aggregate score is higher.
